In [1]:
import pickle, os
import requests as reqlib

import pandas as pd
import numpy as np
import cma

In [2]:
reference_path = "../../results/transit/reference.parquet"
routing_endpoint = "http://localhost:8054/router/transit"
# routing_endpoint = "http://localhost:8029/router/transit"

output_path = "../../results/transit/calibration.p"

# Objective is observation_based or distribution_based
objective = "observation_based"

In [3]:
#if "papermill" in locals():
#    survey_path = papermill.input["survey"]
#    spatial_path = papermill.input["spatial"]

#    routing_endpoint = papermill.params["routing_endpoint"]
#    selected_objective = "individual"

#    progress_path = papermill.output["progress"]
#    output_path = papermill.output["parameters"]

In [4]:
# Load reference data
df_reference = pd.read_parquet(reference_path)

# df_reference = df_reference.iloc[:1000] # For testing

In [5]:
# Identify modes and maximum transfers
modes = [c.replace("legs_", "") for c in df_reference.columns if c.startswith("legs_")]
maximum_transfers = df_reference["transfers"].max()

In [6]:
# Convert to requests
requests = []

for index, row in df_reference.iterrows():
    requests.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": row["departure_time"]
    })

In [7]:
# Prepare querying the routing server
def query_endpoint(requests, utilities):
    response = reqlib.post(routing_endpoint, json = {
        "batch": requests,
        "utilities": utilities
    })

    assert response.status_code == 200

    df_response = { 
        "request_index": [],
        "transfers": []
    }

    for mode in modes:
        df_response["legs_{}".format(mode)] = []

    for row in response.json():
        df_response["request_index"].append(row["request_index"])
        df_response["transfers"].append(np.minimum(row["transfers"], maximum_transfers))
        
        for mode in modes:
            if mode in row["vehicle_legs_by_mode"]:
                df_response["legs_{}".format(mode)].append(row["vehicle_legs_by_mode"][mode])
            else:
                df_response["legs_{}".format(mode)].append(0)
    
    return pd.DataFrame(df_response)

In [8]:
# We need to be careful about the request/response size, so we send individual batches
maximum_batch_size = 4000

def query_endpoint_batched(requests, utilities):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(requests):
        df_response.append(query_endpoint(
            requests[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size],
            utilities))
        
        batch_index += 1
    
    return pd.concat(df_response)

In [9]:
# Define calibration variables
variables = [
    { "name": "rail_u_h", "initial": -1.0 },
    { "name": "subway_u_h", "initial": -1.0, "fixed": True },
    { "name": "bus_u_h", "initial": -1.0 },
    { "name": "tram_u_h", "initial": -1.0 },
    { "name": "other_u_h", "copy": "bus_u_h" },
    { "name": "wait_u_h", "initial": -1.0 },
    { "name": "walk_u_h", "initial": -1.0 },
    { "name": "transfer_u", "initial": -1.0 }
]

In [10]:
# Extend with index information for CMA-ES evaluation
variables_map = { v["name"]: v for v in variables }

active_index = 0

# First find active variables
for variable in variables:
    if "fixed" in variable or "copy" in variable: 
        continue

    variables_map[variable["name"]] = variable
    
    variable["index"] = active_index
    variable["optimized"] = True
    active_index += 1

# Then treat fixed variables and those copying from others
for variable in variables:
    if "fixed" in variable:
        variable["index"] = None
    
    if "copy" in variable:
        assert not "initial" in variable
        variable["initial"] = variables_map[variable["copy"]]["initial"]
        variable["index"] = variables_map[variable["copy"]]["index"]

In [11]:
# Define the optimization objective
modes_weight = 1.0
transfers_weight = 1.0

def calculate_objective_observation_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    df_evaluation["offset"] = transfers_weight * np.abs(
        df_evaluation["transfers_reference"] - df_evaluation["transfers_evaluation"])

    for mode in modes:
        df_evaluation["offset"] += modes_weight * np.abs(
            df_evaluation["legs_{}_reference".format(mode)] - df_evaluation["legs_{}_evaluation".format(mode)]
        )

    return np.sum(df_evaluation["offset"] * df_evaluation["weight"]) / df_evaluation["weight"].sum()

def calculate_objective_distribution_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    reference_mode_distribution = []
    evaluation_mode_distribution = []

    for mode in modes:
        reference_mode_distribution.append((df_evaluation["legs_{}_reference".format(mode)] * df_evaluation["weight"]).sum())
        evaluation_mode_distribution.append((df_evaluation["legs_{}_evaluation".format(mode)] * df_evaluation["weight"]).sum())

    reference_mode_distribution = np.array(reference_mode_distribution) / np.sum(reference_mode_distribution)
    evaluation_mode_distribution = np.array(evaluation_mode_distribution) / np.sum(evaluation_mode_distribution)

    reference_transfer_distribution = []
    evaluation_transfer_distribution = []

    for transfers in range(maximum_transfers + 1):
        f_reference = df_evaluation["transfers_reference"] == transfers
        reference_transfer_distribution.append(df_evaluation.loc[f_reference, "weight"].sum())

        f_evaluation = df_evaluation["transfers_evaluation"] == transfers
        evaluation_transfer_distribution.append(df_evaluation.loc[f_evaluation, "weight"].sum())

    reference_transfer_distribution = np.array(reference_transfer_distribution) / np.sum(reference_transfer_distribution)
    evaluation_transfer_distribution = np.array(evaluation_transfer_distribution) / np.sum(evaluation_transfer_distribution)

    mode_distribution_offset = np.abs(reference_mode_distribution - evaluation_mode_distribution)
    transfer_distribution_offset = np.abs(reference_transfer_distribution - evaluation_transfer_distribution)

    return transfers_weight * np.sum(transfer_distribution_offset) + modes_weight * np.sum(mode_distribution_offset)

In [12]:
# Prepare function to convert CMA-ES' candidate to utilities
def prepare_utilities(values):
    utilities = {}
    
    for variable in variables:
        if variable["index"] is not None:
            utilities[variable["name"]] = values[variable["index"]]
        else:
            utilities[variable["name"]] = variable["initial"]
    
    return utilities

In [13]:
# Prepare bounds and initial values
initial = []
bounds = [[], []]

for variable in variables:
    if "optimized" in variable:
        initial.append(variable["initial"])
        bounds[0].append(-np.inf)
        bounds[1].append(0.0)

In [14]:
# Test connection
assert len(query_endpoint(requests[:5], prepare_utilities(initial))) == 5

In [15]:
# Configure CMA-ES
seed = 1000
sigma = 0.5
iterations = 1000

options = cma.CMAOptions()
options.set("bounds", bounds)
options.set("seed", seed)

algorithm = cma.CMAEvolutionStrategy(initial, sigma, options)

# Load cached data for previous iterations
history = []

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        history = pickle.load(f)

        algorithm.feed_for_resume(
            [h["candidate"] for h in history[1:]], # first one is initial
            [h["objective"] for h in history[1:]]
        )

# Choose objective
if objective == "observation_based":
    calculate_objective = calculate_objective_observation_based
elif objective == "distribution_based":
    calculate_objective = calculate_objective_distribution_based
else:
    raise RuntimeError("Unknown objective")

# Perform a new batch of iterations
for iteration in range(iterations):
    initial_evaluation = len(history) == 0
    candidates = [initial]

    if not initial_evaluation:
        candidates = algorithm.ask()

    objectives = []

    for candidate in candidates:
        utilities = prepare_utilities(candidate)
        df_response = query_endpoint_batched(requests, utilities)
        objective = calculate_objective(df_response)

        objectives.append(objective)

        history.append({
            "candidate": candidate,
            "utilities": utilities,
            "objective": objective,
            "evaluation": df_response,
            "initial": initial_evaluation
        })

    if not initial_evaluation:
        algorithm.tell(candidates, objectives)
        algorithm.disp()

    # Save after a successful CMA-ES iteration
    with open(output_path, "wb+") as f:
        pickle.dump(history, f)

(4_w,9)-aCMA-ES (mu_w=2.8,w_1=49%) in dimension 6 (seed=1000, Thu Jan 23 09:29:28 2025)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      9 1.387276580066681e+00 1.0e+00 4.68e-01  4e-01  5e-01 1:17.2
    2     18 1.189274821747245e+00 1.2e+00 5.23e-01  5e-01  6e-01 2:35.7
    3     27 1.376537753138001e+00 1.5e+00 4.93e-01  4e-01  5e-01 3:54.9
    4     36 1.306592478317635e+00 1.5e+00 4.65e-01  4e-01  5e-01 5:10.6
    5     45 1.266621464219275e+00 1.4e+00 3.85e-01  3e-01  4e-01 6:31.1
    6     54 1.094419992875979e+00 1.5e+00 3.71e-01  3e-01  4e-01 7:53.5
    7     63 1.143632949431935e+00 1.7e+00 3.89e-01  3e-01  4e-01 9:16.9
    8     72 1.163603592176548e+00 1.8e+00 4.13e-01  3e-01  5e-01 10:39.3
    9     81 1.160305977431989e+00 2.0e+00 3.93e-01  3e-01  5e-01 11:57.2
   10     90 1.134895647205012e+00 2.2e+00 3.73e-01  2e-01  4e-01 13:27.6
   11     99 1.128721879911405e+00 2.5e+00 3.42e-01  2e-01  4e-01 14:55.4
   12    108 1.107703478318084e+0

KeyboardInterrupt: 